In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# STEP A: Define the column names (The .txt files don't have them)
columns = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"]

# STEP B: Load your files (Updated with .txt extension)
train_df = pd.read_csv('KDDTrain.txt', names=columns)
test_df = pd.read_csv('KDDTest.txt', names=columns)

# STEP C: Label Encoding (Turning words into numbers)
# We need to encode 'protocol_type', 'service', and 'flag'
categorical_cols = ['protocol_type', 'service', 'flag']
le = LabelEncoder()

for col in categorical_cols:
    # We fit on BOTH train and test to make sure the numbers match
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

# Convert 'label' into 0 (Normal) and 1 (Attack)
train_df['label'] = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['label'] = test_df['label'].apply(lambda x: 0 if x == 'normal' else 1)

# STEP D: Scaling (Making the data professional and balanced)
# We drop 'label' (answer) and 'difficulty' (not needed for AI)
X_train = train_df.drop(['label', 'difficulty'], axis=1)
y_train = train_df['label']

X_test = test_df.drop(['label', 'difficulty'], axis=1)
y_test = test_df['label']

scaler = StandardScaler()

# Fit the scaler on training data and apply to both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Phase 1 Complete!")
print(f"Training shape: {X_train_scaled.shape}")
print(f"Testing shape: {X_test_scaled.shape}")

✅ Phase 1 Complete!
Training shape: (125973, 41)
Testing shape: (22544, 41)


In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# 1. Define the Neural Network Structure
# Think of this as the "architecture" of your AI's brain
model = Sequential([
    # Input Layer + First Hidden Layer (64 neurons)
    # We tell it to look at our 41 columns of data
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2), # This helps the AI not to "over-memorize" so it's better at new data

    # Second Hidden Layer (32 neurons)
    # This helps the AI find deeper patterns
    Dense(32, activation='relu'),

    # Output Layer (1 neuron)
    # 'sigmoid' gives us a probability between 0 and 1
    # 0 = Normal, 1 = Attack
    Dense(1, activation='sigmoid')
])

# 2. Compile the Model
# We tell the AI how to measure its own success
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 3. Start the Training
print("Starting the AI training process...")
history = model.fit(X_train_scaled, y_train,
                    epochs=10,
                    batch_size=32,
                    validation_split=0.2)

print("\n✅ Phase 2 Complete! Your AI is now trained.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Starting the AI training process...
Epoch 1/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.9770 - loss: 0.0657 - val_accuracy: 0.9904 - val_loss: 0.0309
Epoch 2/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - accuracy: 0.9884 - loss: 0.0343 - val_accuracy: 0.9924 - val_loss: 0.0247
Epoch 3/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9903 - loss: 0.0276 - val_accuracy: 0.9929 - val_loss: 0.0228
Epoch 4/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9911 - loss: 0.0251 - val_accuracy: 0.9935 - val_loss: 0.0215
Epoch 5/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.9923 - loss: 0.0224 - val_accuracy: 0.9926 - val_loss: 0.0214
Epoch 6/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9922 - loss: 0.0219 - val_accuracy: 0.9940 - val_loss: 0.0200
Epoch 7/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9930 - loss: 0.0205 - val_accuracy: 0.9946 - val_loss: 0.0197
Epoch 8/10
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step

In [5]:
from sklearn.metrics import classification_report, accuracy_score

# 1. Let the AI predict the answers for the Test Data
predictions = (model.predict(X_test_scaled) > 0.5).astype("int32")

# 2. Compare the AI's answers with the real answers
print("--- FINAL EXAM RESULTS ---")
print(f"Final Accuracy Score: {accuracy_score(y_test, predictions):.4f}")
print("\n--- Detailed Report ---")
print(classification_report(y_test, predictions, target_names=['Normal', 'Attack']))

705/705 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
--- FINAL EXAM RESULTS ---
Final Accuracy Score: 0.7729

--- Detailed Report ---
              precision    recall  f1-score   support

      Normal       0.66      0.98      0.79      9711
      Attack       0.97      0.62      0.76     12833

    accuracy                           0.77     22544
   macro avg       0.82      0.80      0.77     22544
weighted avg       0.84      0.77      0.77     22544



In [6]:
import joblib

# 1. Save the AI Model (The Brain)
# This creates the .h5 file
model.save('nids_model.h5')

# 2. Save the Scaler (The Cleaning Rules)
# This creates the .pkl file
joblib.dump(scaler, 'scaler.pkl')

print("✅ Files Generated!")
print("Now, click the folder icon on the left, then click the 'Refresh' icon (folder with a circle arrow).")

✅ Files Generated!
Now, click the folder icon on the left, then click the 'Refresh' icon (folder with a circle arrow).
